<a href="https://colab.research.google.com/github/zaima-sohail/Speech-Emotion-Recognition/blob/main/Speech_Recognition.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
!pip install kagglehub

In [4]:
import kagglehub
path = kagglehub.dataset_download("dmitrybabko/speech-emotion-recognition-en")
print(path)

100%|██████████| 987M/987M [00:13<00:00, 77.2MB/s]

Extracting files...


/root/.cache/kagglehub/datasets/dmitrybabko/speech-emotion-recognition-en/versions/1


In [5]:
import os

print(os.listdir(path))

for folder, subfolders, files in os.walk(path):
    print(folder, "| subfolders:", len(subfolders), "| files:", len(files))
    print("   example:", files[:3])

['Tess', 'Ravdess', 'Crema', 'Savee']
/root/.cache/kagglehub/datasets/dmitrybabko/speech-emotion-recognition-en/versions/1 | subfolders: 4 | files: 0
   example: []
/root/.cache/kagglehub/datasets/dmitrybabko/speech-emotion-recognition-en/versions/1/Tess | subfolders: 14 | files: 0
   example: []
/root/.cache/kagglehub/datasets/dmitrybabko/speech-emotion-recognition-en/versions/1/Tess/OAF_Fear | subfolders: 0 | files: 200
   example: ['OAF_life_fear.wav', 'OAF_soap_fear.wav', 'OAF_soup_fear.wav']
/root/.cache/kagglehub/datasets/dmitrybabko/speech-emotion-recognition-en/versions/1/Tess/YAF_sad | subfolders: 0 | files: 200
   example: ['YAF_hurl_sad.wav', 'YAF_wire_sad.wav', 'YAF_rag_sad.wav']
/root/.cache/kagglehub/datasets/dmitrybabko/speech-emotion-recognition-en/versions/1/Tess/OAF_neutral | subfolders: 0 | files: 200
   example: ['OAF_pool_neutral.wav', 'OAF_raid_neutral.wav', 'OAF_sure_neutral.wav']
/root/.cache/kagglehub/datasets/dmitrybabko/speech-emotion-recognition-en/versions/

In [6]:
import os

print(os.listdir(path))

for d in os.listdir(path):
    full = os.path.join(path, d)
    if os.path.isdir(full):
        count, examples = 0, []
        for folder, _, files in os.walk(full):
            wavs = [f for f in files if f.endswith(".wav")]
            count += len(wavs)
            if len(examples) < 3:
                examples += wavs[:3 - len(examples)]
        print(d, "| wav files:", count, "| examples:", examples)

['Tess', 'Ravdess', 'Crema', 'Savee']
Tess | wav files: 2800 | examples: ['OAF_life_fear.wav', 'OAF_soap_fear.wav', 'OAF_soup_fear.wav']
Ravdess | wav files: 1440 | examples: ['03-01-08-01-01-01-04.wav', '03-01-02-02-01-02-04.wav', '03-01-04-02-02-02-04.wav']
Crema | wav files: 7442 | examples: ['1083_TAI_DIS_XX.wav', '1084_TSI_SAD_XX.wav', '1004_MTI_DIS_XX.wav']
Savee | wav files: 480 | examples: ['KL_n16.wav', 'JK_n09.wav', 'KL_sa14.wav']


In [7]:
import os, re, pandas as pd

rows = []

# Crema: 1072_ITS_HAP_XX.wav -> actor 1072, emotion HAP
crema = {"ANG":"angry","DIS":"disgust","FEA":"fear","HAP":"happy","NEU":"neutral","SAD":"sad"}
for folder, _, files in os.walk(os.path.join(path, "Crema")):
    for f in files:
        if f.endswith(".wav"):
            p = f.split("_")
            rows.append(("Crema", "Crema_" + p[0], crema.get(p[2]), os.path.join(folder, f)))

# Savee: JK_a07.wav -> speaker JK, emotion code "a"
savee = {"a":"angry","d":"disgust","f":"fear","h":"happy","n":"neutral","sa":"sad","su":"surprise"}
for folder, _, files in os.walk(os.path.join(path, "Savee")):
    for f in files:
        if f.endswith(".wav"):
            spk, code = f.split("_")[0], re.match(r"[a-z]+", f.split("_")[1]).group()
            rows.append(("Savee", "Savee_" + spk, savee.get(code), os.path.join(folder, f)))

# Tess: OAF_keg_ps.wav -> speaker OAF, emotion = last part
tess = {"angry":"angry","disgust":"disgust","fear":"fear","happy":"happy",
        "neutral":"neutral","ps":"surprise","sad":"sad"}
for folder, _, files in os.walk(os.path.join(path, "Tess")):
    for f in files:
        if f.endswith(".wav"):
            p = f[:-4].split("_")
            rows.append(("Tess", "Tess_" + p[0], tess.get(p[-1].lower()), os.path.join(folder, f)))

# Ravdess: 03-01-05-01-02-01-12.wav -> emotion code 3rd, actor last ("calm" hata denge)
rav = {"01":"neutral","02":None,"03":"happy","04":"sad","05":"angry","06":"fear","07":"disgust","08":"surprise"}
for folder, _, files in os.walk(os.path.join(path, "Ravdess")):
    for f in files:
        if f.endswith(".wav"):
            p = f[:-4].split("-")
            rows.append(("Ravdess", "Ravdess_" + p[6], rav.get(p[2]), os.path.join(folder, f)))

df = pd.DataFrame(rows, columns=["dataset", "speaker", "emotion", "path"])
print("Bina label wali files:", df["emotion"].isna().sum())
df = df.dropna(subset=["emotion"]).reset_index(drop=True)

print(len(df), "files,", df["speaker"].nunique(), "speakers")
print(df["emotion"].value_counts())
print(df.groupby("dataset").size())

Bina label wali files: 192
11970 files, 122 speakers
emotion
disgust     1923
sad         1923
angry       1923
happy       1923
fear        1923
neutral     1703
surprise     652
Name: count, dtype: int64
dataset
Crema      7442
Ravdess    1248
Savee       480
Tess       2800
dtype: int64


In [8]:
import numpy as np, librosa
from joblib import Parallel, delayed

def get_features(p):
    audio, sr = librosa.load(p, sr=16000, duration=3)
    mfcc = librosa.feature.mfcc(y=audio, sr=sr, n_mfcc=40)
    return np.hstack([mfcc.mean(axis=1), mfcc.std(axis=1)])   # 80 numbers

X = np.array(Parallel(n_jobs=-1)(delayed(get_features)(p) for p in df["path"]))
y = df["emotion"].values
np.save("X.npy", X)       # save kar lo, dobara nahi nikalna padega
print(X.shape)

(11970, 80)


In [9]:
from sklearn.model_selection import GroupShuffleSplit
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, classification_report

gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
train_idx, test_idx = next(gss.split(X, y, groups=df["speaker"]))

model = make_pipeline(StandardScaler(), SVC(C=10))
model.fit(X[train_idx], y[train_idx])
pred = model.predict(X[test_idx])

print("Accuracy:", accuracy_score(y[test_idx], pred))
print(classification_report(y[test_idx], pred))

Accuracy: 0.48665620094191525
              precision    recall  f1-score   support

       angry       0.67      0.59      0.63       318
     disgust       0.39      0.38      0.38       318
        fear       0.42      0.47      0.44       318
       happy       0.42      0.42      0.42       318
     neutral       0.51      0.46      0.48       282
         sad       0.55      0.59      0.57       318
    surprise       0.42      0.51      0.46        39

    accuracy                           0.49      1911
   macro avg       0.48      0.49      0.48      1911
weighted avg       0.49      0.49      0.49      1911



now train our model by using CNN model

In [10]:
import numpy as np, librosa
from joblib import Parallel, delayed

MAX_LEN = 100

def get_seq(p):
    audio, sr = librosa.load(p, sr=16000, duration=3)
    mfcc = librosa.feature.mfcc(y=audio, sr=sr, n_mfcc=40)      # (40, time)
    if mfcc.shape[1] < MAX_LEN:
        mfcc = np.pad(mfcc, ((0, 0), (0, MAX_LEN - mfcc.shape[1])))
    else:
        mfcc = mfcc[:, :MAX_LEN]
    return mfcc.T                                                # (100, 40)

X_seq = np.array(Parallel(n_jobs=-1)(delayed(get_seq)(p) for p in df["path"]))
np.save("X_seq.npy", X_seq)
print(X_seq.shape)      # (12000 ke aas paas, 100, 40)


(11970, 100, 40)


In [11]:
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()
y_num = le.fit_transform(df["emotion"])
print(le.classes_)

X_tr, X_te = X_seq[train_idx], X_seq[test_idx]
y_tr, y_te = y_num[train_idx], y_num[test_idx]

mean = X_tr.mean(axis=(0, 1))
std  = X_tr.std(axis=(0, 1)) + 1e-8
X_tr = (X_tr - mean) / std
X_te = (X_te - mean) / std

['angry' 'disgust' 'fear' 'happy' 'neutral' 'sad' 'surprise']


In [12]:
from tensorflow.keras import layers, models

n_classes = len(le.classes_)

model = models.Sequential([
    layers.Input(shape=(MAX_LEN, 40)),
    layers.Conv1D(64, 5, padding="same", activation="relu"),
    layers.BatchNormalization(),
    layers.MaxPooling1D(2),
    layers.Dropout(0.3),

    layers.Conv1D(128, 5, padding="same", activation="relu"),
    layers.BatchNormalization(),
    layers.MaxPooling1D(2),
    layers.Dropout(0.3),

    layers.Conv1D(256, 3, padding="same", activation="relu"),
    layers.BatchNormalization(),
    layers.GlobalAveragePooling1D(),
    layers.Dropout(0.4),
    layers.Dense(64, activation="relu"),
    layers.Dense(n_classes, activation="softmax"),
])

model.compile(optimizer="adam",
              loss="sparse_categorical_crossentropy",
              metrics=["accuracy"])

history = model.fit(X_tr, y_tr,
                    validation_data=(X_te, y_te),
                    epochs=40, batch_size=64)

Epoch 1/40
158/158 ━━━━━━━━━━━━━━━━━━━━ 27s 88ms/step - accuracy: 0.4957 - loss: 1.2807 - val_accuracy: 0.4176 - val_loss: 1.4442
Epoch 2/40
158/158 ━━━━━━━━━━━━━━━━━━━━ 1s 6ms/step - accuracy: 0.5866 - loss: 1.0543 - val_accuracy: 0.4144 - val_loss: 1.4192
Epoch 3/40
158/158 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step - accuracy: 0.6215 - loss: 0.9760 - val_accuracy: 0.4024 - val_loss: 1.4419
Epoch 4/40
158/158 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step - accuracy: 0.6415 - loss: 0.9223 - val_accuracy: 0.4322 - val_loss: 1.4085
Epoch 5/40
158/158 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step - accuracy: 0.6653 - loss: 0.8760 - val_accuracy: 0.4668 - val_loss: 1.3596
Epoch 6/40
158/158 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step - accuracy: 0.6762 - loss: 0.8388 - val_accuracy: 0.4642 - val_loss: 1.3779
Epoch 7/40
158/158 ━━━━━━━━━━━━━━━━━━━━ 1s 6ms/step - accuracy: 0.6879 - loss: 0.8204 - val_accuracy: 0.4642 - val_loss: 1.3818
Epoch 8/40
158/158 ━━━━━━━━━━━━━━━━━━━━ 1s 7ms/step - accuracy: 0.6958 - loss: 0.7894 - val_accuracy: 

In [13]:
from sklearn.metrics import classification_report

pred = model.predict(X_te).argmax(axis=1)
print(classification_report(y_te, pred, target_names=le.classes_))

60/60 ━━━━━━━━━━━━━━━━━━━━ 4s 33ms/step
              precision    recall  f1-score   support

       angry       0.78      0.59      0.67       318
     disgust       0.49      0.54      0.51       318
        fear       0.46      0.43      0.44       318
       happy       0.50      0.49      0.49       318
     neutral       0.53      0.51      0.52       282
         sad       0.50      0.65      0.56       318
    surprise       0.83      0.49      0.61        39

    accuracy                           0.53      1911
   macro avg       0.58      0.53      0.55      1911
weighted avg       0.55      0.53      0.54      1911



In [17]:
import json, joblib, numpy as np
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.metrics import classification_report, confusion_matrix

cnn = model                                   # trained CNN
classes = list(le.classes_)
y_test = y[test_idx]                          # string labels

# SVM dobara train (probability ke saath)
svm = make_pipeline(StandardScaler(), SVC(C=10, probability=True))
svm.fit(X[train_idx], y[train_idx])
assert list(svm.classes_) == classes

svm_pred = svm.predict(X[test_idx])
cnn_pred = le.inverse_transform(cnn.predict(X_te).argmax(axis=1))

def pack(pred):
    return {
        "report": classification_report(y_test, pred, labels=classes,
                                        output_dict=True, zero_division=0),
        "cm": confusion_matrix(y_test, pred, labels=classes).tolist(),
    }

metrics = {
    "classes": classes,
    "n_train": int(len(train_idx)), "n_test": int(len(test_idx)),
    "models": {"SVM": pack(svm_pred), "CNN": pack(cnn_pred)},
    "cnn_history": {k: [float(v) for v in vals] for k, vals in history.history.items()},
}
json.dump(metrics, open("metrics.json", "w"))

cnn.save("emotion_cnn.keras")
joblib.dump(svm, "svm.joblib")
np.save("mean.npy", mean); np.save("std.npy", std)
print("Saved. SVM acc:", metrics["models"]["SVM"]["report"]["accuracy"],
      "| CNN acc:", metrics["models"]["CNN"]["report"]["accuracy"])

60/60 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step
Saved. SVM acc: 0.48665620094191525 | CNN acc: 0.5342752485609629


In [18]:
%%writefile app.py
import io, json
import numpy as np, pandas as pd, librosa, joblib
import streamlit as st
import plotly.express as px
import plotly.graph_objects as go
from tensorflow.keras.models import load_model

st.set_page_config(page_title="Speech Emotion Dashboard", page_icon="🎙️", layout="wide")
MAX_LEN = 100
COLORS = {"SVM": "#F59E0B", "CNN": "#6366F1"}

@st.cache_resource
def load_models():
    return (load_model("emotion_cnn.keras"), joblib.load("svm.joblib"),
            np.load("mean.npy"), np.load("std.npy"))

@st.cache_data
def load_metrics():
    return json.load(open("metrics.json"))

cnn, svm, mean, std = load_models()
M = load_metrics()
classes = M["classes"]

def analyse(data):
    y, sr = librosa.load(io.BytesIO(data), sr=16000, duration=3)
    if len(y) < 1600:
        return None
    mfcc = librosa.feature.mfcc(y=y, sr=sr, n_mfcc=40)
    svm_feat = np.hstack([mfcc.mean(axis=1), mfcc.std(axis=1)])[None]
    seq = mfcc[:, :MAX_LEN]
    if seq.shape[1] < MAX_LEN:
        seq = np.pad(seq, ((0, 0), (0, MAX_LEN - seq.shape[1])))
    x = ((seq.T - mean) / std)[None]
    return {"y": y, "sr": sr, "mfcc": mfcc,
            "SVM": svm.predict_proba(svm_feat)[0],
            "CNN": cnn.predict(x, verbose=0)[0]}

# ---------- Sidebar ----------
with st.sidebar:
    st.title("🎙️ SER Dashboard")
    st.caption("Speech Emotion Recognition")
    st.write(f"**Train clips:** {M['n_train']}")
    st.write(f"**Test clips:** {M['n_test']}")
    st.write(f"**Emotions:** {len(classes)}")
    st.info("Test set mein woh speakers hain jo training mein nahi the.")

st.title("Speech Emotion Recognition")
tab1, tab2, tab3, tab4 = st.tabs(
    ["🎤 Live Prediction", "📊 Model Comparison", "🔍 Per-Emotion Analysis", "🧩 Confusion Matrix"])

# ---------- Tab 1: Live prediction ----------
with tab1:
    c1, c2 = st.columns(2)
    data = None
    with c1:
        up = st.file_uploader("WAV file upload karo", type=["wav"])
        if up is not None:
            data = up.getvalue()
    with c2:
        if hasattr(st, "audio_input"):
            rec = st.audio_input("Ya apni awaaz record karo")
            if rec is not None:
                data = rec.getvalue()
        else:
            st.warning("Is Streamlit version mein mic recording nahi hai. "
                       "`pip install -U streamlit` chalao.")

    if data:
        st.audio(data)
        try:
            r = analyse(data)
        except Exception as e:
            r = None
            st.error(f"Audio process nahi hui: {e}")
        if r is None:
            st.warning("Audio bahut chhoti hai, dobara record karo (kam az kam 1 second).")
        else:
            m1, m2 = st.columns(2)
            for col, name in zip((m1, m2), ("SVM", "CNN")):
                p = r[name]; i = int(np.argmax(p))
                col.metric(f"{name} prediction", classes[i].upper(), f"{p[i]*100:.1f}% confidence")

            df_p = pd.DataFrame({"emotion": classes * 2,
                                 "probability": np.concatenate([r["SVM"], r["CNN"]]),
                                 "model": ["SVM"] * len(classes) + ["CNN"] * len(classes)})
            fig = px.bar(df_p, x="emotion", y="probability", color="model", barmode="group",
                         color_discrete_map=COLORS, title="Probability per emotion")
            st.plotly_chart(fig, use_container_width=True)

            w1, w2 = st.columns(2)
            t = np.arange(len(r["y"])) / r["sr"]
            fw = go.Figure(go.Scatter(x=t[::20], y=r["y"][::20], line=dict(color="#6366F1")))
            fw.update_layout(title="Waveform", xaxis_title="seconds", height=320)
            w1.plotly_chart(fw, use_container_width=True)
            fm = px.imshow(r["mfcc"], aspect="auto", origin="lower",
                           color_continuous_scale="Viridis", title="MFCC heatmap")
            fm.update_layout(height=320)
            w2.plotly_chart(fm, use_container_width=True)

# ---------- Tab 2: Model comparison ----------
with tab2:
    rows = []
    for name, d in M["models"].items():
        rep = d["report"]
        rows.append({"Model": name, "Accuracy": rep["accuracy"],
                     "Macro F1": rep["macro avg"]["f1-score"],
                     "Macro Precision": rep["macro avg"]["precision"],
                     "Macro Recall": rep["macro avg"]["recall"]})
    comp = pd.DataFrame(rows)

    cols = st.columns(len(comp))
    for col, (_, row) in zip(cols, comp.iterrows()):
        col.metric(f"{row['Model']} accuracy", f"{row['Accuracy']*100:.1f}%")
    if len(comp) == 2:
        diff = (comp.loc[1, "Accuracy"] - comp.loc[0, "Accuracy"]) * 100
        st.success(f"{comp.loc[1,'Model']} ki accuracy {comp.loc[0,'Model']} se {diff:+.1f} points ka farq rakhti hai.")

    long = comp.melt(id_vars="Model", var_name="Metric", value_name="Score")
    fig = px.bar(long, x="Metric", y="Score", color="Model", barmode="group",
                 color_discrete_map=COLORS, title="Overall metrics")
    st.plotly_chart(fig, use_container_width=True)
    st.dataframe(comp.style.format({c: "{:.3f}" for c in comp.columns[1:]}),
                 use_container_width=True)

    h = M.get("cnn_history", {})
    if "accuracy" in h:
        fh = go.Figure()
        fh.add_scatter(y=h["accuracy"], name="train accuracy", line=dict(color="#6366F1"))
        if "val_accuracy" in h:
            fh.add_scatter(y=h["val_accuracy"], name="test accuracy", line=dict(color="#F59E0B"))
        fh.update_layout(title="CNN training curve", xaxis_title="epoch", yaxis_title="accuracy")
        st.plotly_chart(fh, use_container_width=True)

# ---------- Tab 3: Per-emotion ----------
with tab3:
    rows = []
    for name, d in M["models"].items():
        for c in classes:
            rows.append({"Model": name, "Emotion": c, "F1": d["report"][c]["f1-score"],
                         "Support": d["report"][c]["support"]})
    pe = pd.DataFrame(rows)
    fig = px.bar(pe, x="Emotion", y="F1", color="Model", barmode="group",
                 color_discrete_map=COLORS, title="F1-score per emotion")
    st.plotly_chart(fig, use_container_width=True)
    sup = pe[pe["Model"] == pe["Model"].iloc[0]][["Emotion", "Support"]]
    st.plotly_chart(px.bar(sup, x="Emotion", y="Support", title="Test clips per emotion"),
                    use_container_width=True)
    st.caption("Jis emotion ke test clips kam hon, uska F1 kam bharosemand hota hai.")

# ---------- Tab 4: Confusion matrix ----------
with tab4:
    pick = st.radio("Model", list(M["models"].keys()), horizontal=True)
    cm = np.array(M["models"][pick]["cm"], dtype=float)
    cmn = cm / cm.sum(axis=1, keepdims=True).clip(min=1)
    fig = px.imshow(cmn, x=classes, y=classes, text_auto=".2f",
                    color_continuous_scale="Blues", aspect="auto",
                    labels=dict(x="Predicted", y="Actual", color="Fraction"),
                    title=f"{pick} confusion matrix (row-normalized)")
    fig.update_layout(height=600)
    st.plotly_chart(fig, use_container_width=True)

Overwriting app.py


In [27]:
resources_text = '''

# ---------- Resources ----------
with st.expander("📚 Resources & Dataset Information"):
    st.markdown("""
    ### Speech Emotion Recognition (SER)
    This application evaluates two models (Classical Machine Learning with SVM and Deep Learning with a 1D CNN) using MFCC features.

    **Datasets Included:**
    * **CREMA-D** (Crowd-sourced Emotional Multimodal Actors Dataset)
    * **RAVDESS** (Ryerson Audio-Visual Database of Emotional Speech and Song)
    * **TESS** (Toronto Emotional Speech Set)
    * **SAVEE** (Surrey Audio-Visual Ecstatically Expressive)

    **References:**
    * Dataset source: [Kaggle Speech Emotion Recognition EN](https://www.kaggle.com/datasets/dmitrybabko/speech-emotion-recognition-en)
    * Developed using TensorFlow/Keras, Scikit-Learn, Librosa, and Streamlit.
    """)
'''

with open("app.py", "a") as f:
    f.write(resources_text)
print("Resources section successfully appended to app.py!")

Resources section successfully appended to app.py!


In [26]:
# Restart Streamlit and Cloudflared to apply the updated app.py
import os, time
os.system("pkill -f streamlit")
os.system("pkill -f cloudflared")
time.sleep(2)

import subprocess
st_log = open("st.log", "w")
subprocess.Popen(["streamlit", "run", "app.py",
                  "--server.port", "8501", "--server.headless", "true"],
                 stdout=st_log, stderr=subprocess.STDOUT)
time.sleep(5)
print("Streamlit restarted with the updated app.py!")

Streamlit restarted with the updated app.py!


In [28]:
!pip install -q -U streamlit plotly joblib
!pkill -f streamlit
!pkill -f cloudflared

In [29]:
import subprocess, time, re, os

# Purani app aur tunnel band karo
os.system("pkill -f streamlit")
os.system("pkill -f cloudflared")
time.sleep(2)

# cloudflared download (sirf pehli baar)
if not os.path.exists("cloudflared"):
    os.system("wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O cloudflared")
    os.system("chmod +x cloudflared")

# Streamlit background mein chalao (logs st.log mein)
st_log = open("st.log", "w")
subprocess.Popen(["streamlit", "run", "app.py",
                  "--server.port", "8501", "--server.headless", "true"],
                 stdout=st_log, stderr=subprocess.STDOUT)
time.sleep(10)

# Tunnel chalao (logs cf.log mein)
cf_log = open("cf.log", "w")
subprocess.Popen(["./cloudflared", "tunnel", "--url", "http://localhost:8501"],
                 stdout=cf_log, stderr=subprocess.STDOUT)

# Link dhoondo
link = None
for _ in range(30):
    time.sleep(2)
    m = re.search(r"https://[-a-z0-9]+\.trycloudflare\.com", open("cf.log").read())
    if m:
        link = m.group()
        break

print("Dashboard link:", link if link else "link nahi mila, neeche logs dekho")

Dashboard link: https://outlined-missile-fellowship-watt.trycloudflare.com


In [21]:
import numpy as np
from IPython.display import Audio, display

idx = np.random.choice(len(test_idx), 8, replace=False)
pred = model.predict(X_te[idx], verbose=0).argmax(axis=1)   # model = CNN

for i, p in zip(idx, pred):
    path_i = df["path"].iloc[test_idx[i]]
    actual, guess = le.classes_[y_te[i]], le.classes_[p]
    print(("✅" if actual == guess else "❌"), "actual:", actual, "| predicted:", guess)
    display(Audio(path_i))

✅ actual: sad | predicted: sad


✅ actual: neutral | predicted: neutral


✅ actual: happy | predicted: happy


✅ actual: happy | predicted: happy


❌ actual: disgust | predicted: sad


❌ actual: surprise | predicted: happy


❌ actual: neutral | predicted: sad


❌ actual: sad | predicted: happy


In [22]:
from google.colab import files
files.download(df["path"].iloc[test_idx[0]])

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [24]:
# Note: Streamlit functions like st.file_uploader cannot be executed directly in Colab cells.
# They must reside in 'app.py' and are served via the background Streamlit process.